# 13 — Cluster Descriptions, Category Mapping, and Semantic Evaluation

Three sections:
1. **LexRank descriptions** — for every clustering method × cluster, summarise
   all articles in the cluster to 2-3 representative sentences using LexRank.
2. **Semantic category mapping** — embed each cluster description with
   `all-mpnet-base-v2`, find the most similar of the 16 canonical category
   descriptions by cosine similarity.
3. **Semantic evaluation** — report a per-method semantic similarity score
   (mean cosine similarity across all 16 clusters) alongside the existing
   Hungarian-ACC, producing a combined comparison table.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

from utils import config
from utils.cluster_description import generate_cluster_descriptions

train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")
summaries   = train_clean["summary"].tolist()

# Discover all clustering methods that have saved label arrays
label_files = sorted(config.RESULTS_DIR.glob("cluster_labels_*.npy"))
ALL_METHODS = [f.stem.replace("cluster_labels_", "") for f in label_files]
print(f"Found {len(ALL_METHODS)} clustering methods:")
for m in ALL_METHODS:
    print(f"  {m}")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# SECTION 1 — LexRank cluster descriptions
# ════════════════════════════════════════════════════════════════════
all_descriptions = {}   # method -> {cluster_id(int): description(str)}

for method in ALL_METHODS:
    labels_path = config.RESULTS_DIR / f"cluster_labels_{method}.npy"
    cluster_labels = np.load(labels_path)

    descs = generate_cluster_descriptions(summaries, cluster_labels, n_sentences=3, min_words=10)
    all_descriptions[method] = descs

    # Persist as JSON with string keys for portability
    out_path = config.RESULTS_DIR / f"cluster_descriptions_{method}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in descs.items()}, f, indent=2, ensure_ascii=False)

    print(f"  {method}: {len(descs)} cluster descriptions saved → {out_path.name}")

print(f"\nSection 1 complete. {len(ALL_METHODS)} methods processed.")

In [ ]:
# Print a sample description for cluster 0 of each method
print("=== Sample descriptions (cluster 0 per method) ===\n")
for method in ALL_METHODS[:5]:   # show first 5 to avoid scrolling
    d = all_descriptions[method].get(0, "(missing)")
    print(f"[{method}]\n{d}\n")